In [ ]:
from pathlib import Path

import librosa
import numpy as np
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# код из 4 лабы

In [ ]:
def mfcc_vad(wav: np.ndarray, sr: int = 16_000, n_expected: int = 8, threshold: float = 0.57, counter: int = 0) -> list:
    win_length = int(0.025 * sr)
    hop_length = int(0.010 * sr)

    # считаем энергию c MFCC, почти то же самое, что было во второй лабе
    mfcc = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=13, n_fft=win_length, hop_length=hop_length, win_length=win_length)
    energy = mfcc[0]

    energy_norm = (energy - np.mean(energy)) / (np.std(energy) + 1e-6)
    detected = energy_norm > threshold

    frame_times = librosa.frames_to_time(np.arange(len(detected)), sr=sr, hop_length=hop_length)

    segments = []
    start_time = None

    for i, active in enumerate(detected):
        if active and start_time is None:
            start_time = frame_times[i]
        elif not active and start_time is not None:
            end_time = frame_times[i]
            segments.append((int(start_time * sr), int(end_time * sr)))
            start_time = None

    # дошли до конца, трекая аудио
    if start_time is not None:
        segments.append((int(start_time * sr), len(wav)))

    if len(segments) > n_expected:
        # сортируем по продолжительности и по началу аудио
        print(f"Перебрали. {len(segments)} сегментов")
        counter += 1
        segments = sorted(segments, key=lambda s: s[1] - s[0], reverse=True)[:n_expected]
        segments = sorted(segments, key=lambda s: s[0])

    if len(segments) != n_expected:
        print(f"Недобрали. {len(segments)} сегментов")
        counter += 1

    return segments[:n_expected], counter

# Меняем классификатор на QDA

In [ ]:
def mean_mfcc_features(y: np.ndarray, sr: int, n_mfcc: int = 13, n_fft: int = 512, hop_length: int = 160) -> np.ndarray:
    y, _ = librosa.effects.trim(y, top_db=20)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    return np.mean(mfcc, axis=1)


def preprocess(data_dir: int) -> np.ndarray:
    X, y = [], []
    counter = 0
    data_dir = Path(data_dir)

    for wav_path in data_dir.glob("*.wav"):
        stem = wav_path.stem
        try:
            true_labels = [int(bit) for bit in stem.split("_")]
            assert len(true_labels) == 8, wav_path
        except AssertionError:
            continue

        wav, sr = librosa.load(wav_path, sr=None)
        segments, counter = mfcc_vad(wav, sr, n_expected=8, counter=counter)

        for i, (start, end) in enumerate(segments):
            seg_audio = wav[start:end]
            if len(seg_audio) == 0:
                continue
            feat = mean_mfcc_features(seg_audio, sr)
            X.append(feat)
            y.append(true_labels[i])
    print(f"VAD ошибся в {counter} файлах")

    return np.array(X), np.array(y)


def classifier(X: np.ndarray, y: np.ndarray) -> None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    qda = QDA()
    qda.fit(X_train_scaled, y_train)

    # Evaluate
    y_pred = qda.predict(X_test_scaled)
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["no", "yes"]))


X, y = preprocess("waves_yesno/")

print(f"Распределение классов: no={np.sum(y == 0)}, yes={np.sum(y == 1)}")

classifier(X, y)

/home/artyom/myprojects/ITMO/DSP/.venv/lib/python3.12/site-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


Перебрали. 11 сегментов
Перебрали. 9 сегментов
Перебрали. 12 сегментов
Перебрали. 10 сегментов
Перебрали. 11 сегментов
Перебрали. 12 сегментов
Перебрали. 9 сегментов
Перебрали. 10 сегментов
Перебрали. 11 сегментов
Перебрали. 9 сегментов
Перебрали. 9 сегментов
Перебрали. 10 сегментов
Перебрали. 9 сегментов
Перебрали. 10 сегментов
Перебрали. 10 сегментов
Перебрали. 13 сегментов
Перебрали. 9 сегментов
Перебрали. 9 сегментов
Перебрали. 10 сегментов
Перебрали. 9 сегментов
Перебрали. 15 сегментов
Перебрали. 10 сегментов
Перебрали. 9 сегментов
Перебрали. 9 сегментов
Перебрали. 9 сегментов
Недобрали. 7 сегментов


/home/artyom/myprojects/ITMO/DSP/.venv/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=240
  warnings.warn(


Перебрали. 9 сегментов
Перебрали. 10 сегментов
Перебрали. 11 сегментов
Перебрали. 9 сегментов
VAD ошибся в 30 файлах
Распределение классов: no=228, yes=251
Classification Report:
              precision    recall  f1-score   support

          no       0.98      0.98      0.98        46
         yes       0.98      0.98      0.98        50

    accuracy                           0.98        96
   macro avg       0.98      0.98      0.98        96
weighted avg       0.98      0.98      0.98        96



# Вывод

Код с новым классификатором работает